In [1]:
#Imported Libraries
from selenium import webdriver
import time
import pandas as pd
import os 
import numpy as np

In [2]:
#Import Packages From Selenium
from selenium.webdriver.support.select import Select 
from selenium.webdriver.support.ui import WebDriverWait    
from selenium.webdriver.common.by import By    
from selenium.webdriver.support import expected_conditions as EC  
from selenium.webdriver.common.keys import Keys

In [3]:
from selenium import webdriver

cService = webdriver.ChromeService(executable_path='C:/Webdriver/chromedriver.exe')
driver = webdriver.Chrome(service = cService)

## Opens the chrome and take login page of Linkedin

In [14]:
  #I can control chrome with driver variable;
driver.get("https://www.linkedin.com/login?fromSignIn=true&trk=guest_homepage-basic_nav-header-signin")                             # Gets to linkedin home page;
driver.maximize_window()                                            # Maximises the present browser or page window;

# Automatic Login- Linkedin 

In [15]:
username = driver.find_element(By.ID, "username")
password = driver.find_element(By.ID, "password")
username.send_keys("mkmiglani88@gmail.com")                                   # Founder the username box as above and sented values to that particular box;
password.send_keys("Iamthewarrior@143")
driver.find_element(By.CLASS_NAME,"btn__primary--large").click()    # Finding submit button and clicking it;

Direct link for job search in linkedin:

In [16]:
driver.get("https://www.linkedin.com/jobs/search/?currentJobId=3970847943&keywords=sales%20and%20marketing&origin=JOBS_HOME_KEYWORD_AUTOCOMPLETE&refresh=true")
# Getting straight to the page where I have already searched "Data Analyst" in the search bar;

Function to find all company and their linkedin page links for getting number of followers:

In [17]:
def job_exist_function(): # I think this function gets all company names;
    company_names= []
    n=26
    try:
        for i in range(n):
            company=driver.find_elements(By.CLASS_NAME, 'job-card-container__company-name')[i].text     #Company name;
            company_names.append(company)
    except:
        return company_names

Function to scroll down the page till end:

In [18]:
def scroll_till_down(): # This function will scroll down and looks for all the jobs in one page;
    Assumed_company_num=26
    try:
        for num in range(Assumed_company_num):
            search=driver.find_elements(By.CLASS_NAME, 'job-card-container__company-name')[num] 
            driver.execute_script('arguments[0].scrollIntoView({behavior: "auto", block: "start", inline: "end"});', search)   
            
    except:
        pass

Function to move through all pages till p-1 pages:

In [19]:
def Checks_if_page_exist_and_clicks(p):
    s=driver.find_elements(By.CLASS_NAME,'artdeco-pagination__indicator--number') # List of pages buttons;
    for num in range(len(s)):   # Taking the buttons one at a time;
        if s[num].text==str(p): # 
            print(f"Got page {s[num].text}")
            time.sleep(1)
            s[num].click()
            break
        if num==9: # If at all the loop reach last one or 9th one press'...', but in one case it happens;
            print('Was inside')
            s[8].click() # Because 8th one is '...';
            s=driver.find_elements(By.CLASS_NAME,'artdeco-pagination__indicator--number') # List of pages buttons;
            for num in range(len(s)):
                if s[num].text==str(p):
                    print(f"Got page {s[num].text}")
                    s[num].click()
                    break

Function to extract information about the job profile in a particular linkedin:

In [20]:
def scrape_job_data():     # Function two
    job_name = driver.find_element(By.CLASS_NAME,"jobs-unified-top-card__job-title").text
    company_location = driver.find_elements(By.CLASS_NAME,"jobs-unified-top-card__bullet")[0].text
    job_type_and_level = driver.find_elements(By.CLASS_NAME, "jobs-unified-top-card__job-insight")[0].text
    employees_and_industry = driver.find_elements(By.CLASS_NAME, "jobs-unified-top-card__job-insight")[1].text
    company_name = driver.find_element(By.CSS_SELECTOR,'.ember-view.t-black.t-normal').text
    link = driver.find_element(By.CSS_SELECTOR,'.ember-view.t-black.t-normal').get_attribute('href')
    job_description = driver.find_element(By.CLASS_NAME, "jobs-description-content__text--stretch").text
    try:
        job_applicants = driver.find_element(By.CLASS_NAME, "jobs-unified-top-card__applicant-count").text
    except:
        job_applicants = "Nil"
    return [company_name, job_name, company_location, job_applicants, job_type_and_level, employees_and_industry, link, job_description]


Function to get number of followers from each companie's Linkedin page:

In [21]:
def scrape_company_followers():     # There is a chance of error occuring in followers but if error occurs then followers will be empty
    following = ""
    following = driver.find_elements(By.CLASS_NAME, "org-top-card-summary-info-list__info-item")[-1].text
    Industry = driver.find_elements(By.CLASS_NAME, "org-top-card-summary-info-list__info-item")[0].text
    return following,Industry

# Extracting all the information about the job-->

In [23]:
page=100
df = pd.DataFrame(columns = ["Company","Job_name", "Location","Applicants", "Job_level", "Employees","Link","Job_description"])
company_list=[]
company_links=[]
try:

        for p in range(1,page):
                Checks_if_page_exist_and_clicks(p)
                scroll_till_down()
                names=job_exist_function()
                total_job_num=len(names)        # Total jobs in a particular page;

                for num in range(total_job_num):
                        driver.find_elements(By.CLASS_NAME,'job-card-list__title')[num].click() # Clicks the particular job profile;
                        try:
                                job_details = scrape_job_data() # Some details about job;
                        except:
                                time.sleep(1.5)
                                try:
                                        job_details = scrape_job_data()
                                except:
                                        continue
                        print(f"Success got job: {num} of page: {p}")
                        df.loc[len(df)] = job_details
except:
        print("Error Occured !")

Got page 1
Got page 2
Got page 3
Got page 4
Got page 5
Got page 6
Got page 7
Got page 8
Was inside
Got page 9
Error Occured !


Saving DataFrame

In [32]:
df.to_csv('saved_for_safety.csv')

From above DF taking Company names and Company Linkedin page links:

In [24]:
company_names = df.Company.values.tolist()
company_links = df.Link.values.tolist()
company_df = pd.DataFrame(data={'Company':company_names, 'links':company_links}) 

company_df=company_df.drop_duplicates()                                         # Removed duplicates
company_df.reset_index(drop=True,inplace=True)
company_df.dropna(axis=0, inplace=True)

Extracting number of Followers & Industry of a particular company:

In [25]:
followers_list = []
industry_list = []
for num, link in enumerate(company_df.links.values):
    driver.get(link)
    try:
        followers, industry = scrape_company_followers()
    except:
        time.sleep(1.5)
        try:
            followers, industry = scrape_company_followers()
        except:
            followers, industry = np.nan, np.nan
    followers_list.append(followers)
    industry_list.append(industry)

Converting to a DF:

In [26]:
company_df['Followers'] = followers_list
company_df['Industry'] = industry_list
company_df = company_df[['Company','Followers','Industry']]
company_df

,Company,Followers,Industry


# Joined Followers df and Job_details df

In [30]:
new_df=df.join(company_df.set_index(keys='Company'), on='Company')
new_df.isnull().sum()
new_df.dropna(axis=0,inplace=True)
new_df.reset_index(inplace=True, drop=True)
new_df

ValueError: You are trying to merge on object and float64 columns. If you wish to proceed you should use pd.concat

Saving Final DF:

In [31]:
new_df.to_csv('New_uncleaned_data.csv')

NameError: name 'new_df' is not defined